In [1]:
import sqlite3
import pandas as pd

In [2]:
db = sqlite3.connect("../data/database/collected.db")

In [3]:
df = pd.read_sql_query(
    """
    SELECT *
    FROM  messages 
    """,
    db
)

## Cleaning Data

In [ ]:
df.drop(df[df['caption_path'].isna()&df['media_path'].notna()].index, inplace=True)
df.drop(df[(df['source_name']== "شمسی نیوز") & df['media_path'].notna()].index, inplace=True)
df.drop(columns=['edit_date','caption_path'],inplace=True)
df = df[df["text"].notna() & (df["text"].str.strip() != "")].copy()


In [ ]:
import re

def normalize_text(text):
    text = str(text)

    # Arabic characters -> Persian
    text = text.replace("ي", "ی").replace("ك", "ک")

    # Normalize spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Too many blank lines -> max one blank line
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()


df["content"] = df["text"].apply(normalize_text)

## split groups and channels 

In [6]:
channels = df[df["source_type"] == "channel"].copy()

groups = df[df["source_type"] == "group"].copy()

In [7]:
groups["source_id"] = pd.to_numeric(
    groups["source_id"],
    errors="coerce"
).astype("Int64")

groups["message_id"] = pd.to_numeric(
    groups["message_id"],
    errors="coerce"
).astype("Int64")

groups["reply_to_message_id"] = pd.to_numeric(
    groups["reply_to_message_id"],
    errors="coerce"
).astype("Int64")

In [8]:
parent_map = {}

for row in groups.itertuples():
    key = (int(row.source_id), int(row.message_id))

    if pd.notna(row.reply_to_message_id):
        value = int(row.reply_to_message_id)
    else:
        value = None

    parent_map[key] = value


def find_root(source_id, message_id):
    source_id = int(source_id)
    current = int(message_id)
    visited = set()

    while True:
        key = (source_id, current)

        if key in visited:
            break

        visited.add(key)

        parent = parent_map.get(key)

        if parent is None:
            break

        if (source_id, parent) not in parent_map:
            break

        current = parent

    return current


groups["root_message_id"] = [
    find_root(source_id, message_id)
    for source_id, message_id
    in zip(groups["source_id"], groups["message_id"])
]

## Create Documents

In [9]:
channel_documents = channels[
    [
        "source_id",
        "source_name",
        "message_id",
        "message_date",
        "content",
        "media_path"
    ]
].copy()

channel_documents["document_type"] = "channel_message"

channel_documents["document_id"] = (
    channel_documents["source_id"].astype(str)
    + "_"
    + channel_documents["message_id"].astype(str)
)

In [10]:
def build_thread_document(thread):
    thread = thread.sort_values("message_date")

    parts = []

    for row in thread.itertuples():
        parts.append(
            f"[message {row.message_id}] {row.content}"
        )

    return "\n".join(parts)

In [11]:
group_documents = (
    groups
    .groupby(
        ["source_id", "source_name", "root_message_id"],
        as_index=False
    )
    .apply(
        lambda thread: pd.Series({
            "content": build_thread_document(thread),

            "message_ids": thread["message_id"].tolist(),

            "media_paths": (
                thread["media_path"]
                .dropna()
                .tolist()
            ),

            "message_date": thread["message_date"].min(),

            "message_count": len(thread)
        }),
        include_groups=False
    )
    .reset_index(drop=True)
)

In [12]:
group_documents["document_type"] = "group_thread"

group_documents["document_id"] = (
    group_documents["source_id"].astype(str)
    + "_thread_"
    + group_documents["root_message_id"].astype(str)
)

In [13]:
channel_final = channel_documents.copy()

channel_final["message_ids"] = channel_final["message_id"].apply(
    lambda x: [int(x)]
)

channel_final["media_paths"] = channel_final["media_path"].apply(
    lambda x: [] if pd.isna(x) else [x]
)

channel_final = channel_final[
    [
        "document_id",
        "document_type",
        "source_id",
        "source_name",
        "message_date",
        "content",
        "message_ids",
        "media_paths"
    ]
]


group_final = group_documents[
    [
        "document_id",
        "document_type",
        "source_id",
        "source_name",
        "message_date",
        "content",
        "message_ids",
        "media_paths"
    ]
].copy()


documents = pd.concat(
    [channel_final, group_final],
    ignore_index=True
)

print("Documents:", len(documents))
documents.head()

Documents: 66192


,document_id,document_type,source_id,source_name,message_date,content,message_ids,media_paths
0,-1002399800721_122,channel_message,-1002399800721,اطلاع‌رسانی دفتر استادان,2024-10-03T16:27:17+00:00,. \n🟦🟦 قابل توجه دانشجویان گرامی 🟦🟦\n\n🟩🟩🟩🟩🟩🟩🟩...,[122],[]
1,-1002399800721_123,channel_message,-1002399800721,اطلاع‌رسانی دفتر استادان,2024-10-03T16:27:17+00:00,. \n🟦🟦 قابل توجه دانشجویان گرامی 🟦🟦\n\n🟩🟩🟩🟩🟩🟩🟩...,[123],[]
2,-1002399800721_124,channel_message,-1002399800721,اطلاع‌رسانی دفتر استادان,2024-10-03T16:27:18+00:00,. \n🟦🟦 قابل توجه دانشجویان گرامی 🟦🟦\n\n🟩🟩🟩🟩🟩🟩🟩...,[124],[]
3,-1002399800721_125,channel_message,-1002399800721,اطلاع‌رسانی دفتر استادان,2024-10-03T16:27:18+00:00,. \n🟦🟦 قابل توجه دانشجویان گرامی 🟦🟦\n\n🟩🟩🟩🟩🟩🟩🟩...,[125],[]
4,-1002399800721_126,channel_message,-1002399800721,اطلاع‌رسانی دفتر استادان,2024-10-03T16:27:18+00:00,. \n🟦🟦 قابل توجه دانشجویان گرامی 🟦🟦\n\n🟩🟩🟩🟩🟩🟩🟩...,[126],[]


In [ ]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True,exist_ok=True)

documents.to_json(output_dir / "documents.jsonl",orient="records",lines=True,force_ascii=False)

print(f"Saved: {output_dir}/documents.jsonl")

Saved: ..\data\processed/documents.json


## Now our data is ready